# A08 · Multimodal: Vision, Documents & Audio

**Track:** Advanced (lesson 8 of 10) · **Time:** ~2.5 h · **Mode:** runs OFFLINE for free, or LIVE with `ANTHROPIC_API_KEY`
**Prereqs:** A01 (LLM APIs), A05 (structured outputs) · **You will build:** a receipt-extraction pipeline with a schema, validation, confidence routing and a human-review queue

---
## 🎯 Why this matters
Most business knowledge is not clean text. It is a photo of a receipt, a scanned invoice, a PDF with a table, a recorded call. A model that can *look* at a page removes the brittle
"OCR → regex → hope" pipeline, but only if you treat its output like any other untrusted input: **extract into a schema, validate, and route doubt to a human.**

**Real scenario.** *Kiwi Field Services* reimburses staff expenses. 3,000 receipt photos arrive each month; two finance clerks type them in. Goal: auto-process the receipts the
system is sure about, and queue the rest for a clerk, with **no wrong number silently reaching the ledger** that a cheap check could have caught.

## 🧭 Learning objectives
By the end you can:
1. Show how images and PDFs are passed to a model (content blocks) and estimate the token cost of an image.
2. Extract structured data from document images into a validated pydantic schema, with a retry-on-invalid loop.
3. Design **confidence routing**: cross-field consistency checks + a threshold that sends uncertain results to human review, and measure the trade-off.
4. Choose between classic OCR and native vision for a given document type, and say where each fails.
5. Compute word error rate (WER) for ASR and place ASR/TTS in a voice pipeline.

In [ ]:
# Setup: runs LIVE with ANTHROPIC_API_KEY, otherwise OFFLINE (scripted replay, free)
import sys, pathlib, subprocess
_find = lambda: next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "agentkit").is_dir()), None)
_root = _find()
if _root is None:                        # Colab / standalone download: fetch the repo helper package
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/nexageapps/AI.git", "_ai_repo"], check=False)
    _root = pathlib.Path("_ai_repo").resolve()
sys.path.insert(0, str(_root))
from agentkit import setup
setup()

## 1 · Concept: how a model "sees"

A vision-capable model takes **content blocks** instead of a plain string. An image is just another block in the user turn, next to your text instructions. The shapes below are the
Messages-API content-block format (shown as inert data, not sent anywhere). Check the current docs for supported media types and size limits.

```python
# one user turn = a list of blocks
{"role": "user", "content": [
    {"type": "image",    "source": {"type": "base64", "media_type": "image/png",       "data": "<base64 bytes>"}},
    {"type": "text",     "text": "Extract the receipt as JSON ..."},
]}

# a PDF goes in a 'document' block; the model reads text AND the page images
{"type": "document", "source": {"type": "base64", "media_type": "application/pdf", "data": "<base64 bytes>"}}

# images/documents can also be referenced by URL source or by an uploaded-file id (see the docs)
```

**Cost model.** Images are billed as input tokens. The docs give a rule of thumb of roughly `width × height / 750` tokens per image, and very large images are downscaled before the
model sees them (so pixels beyond the cap buy you cost but not detail). Treat that as an estimate and confirm in the current docs.

**Mental model.** The model does not run a separate OCR engine and then read the text. The image is encoded into the same token stream as your words, so it can use layout,
logos, handwriting and context ("a total at the bottom right is probably the total") jointly. That is why it often beats OCR on messy documents, and also why its mistakes are
*plausible-looking* rather than garbled.

> **Question to hold:** if its mistakes look plausible, how will you ever catch them? (Answer: redundancy in the document itself, e.g. line items must sum to the total, plus calibrated doubt.)

## 2 · The data: synthetic receipts with ground truth
We cannot ship real receipts, so we **render** them with matplotlib into pixel arrays, keeping the true fields next to each image. Knowing the truth is what lets us *measure* the
pipeline instead of eyeballing it. NZ receipts show GST (15%) as a component of a GST-inclusive total, so `gst = total × 3/23`.

In [ ]:
import io, json, base64, random, re
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

rng = random.Random(8)
MERCHANTS = {"Kowhai Cafe": [("Flat white", 5.2), ("Ham toastie", 9.5), ("Almond croissant", 6.8), ("Long black", 4.8)],
             "Tui Hardware": [("Cable ties 100pk", 8.9), ("Work gloves", 17.5), ("Duct tape", 7.4), ("Tape measure 5m", 14.9)],
             "Ranui Pharmacy": [("Sunscreen SPF50", 18.9), ("Plasters 40pk", 6.5), ("Hand sanitiser", 5.9)],
             "Harbour Taxis": [("Fare CBD to Airport", 62.0), ("Booking fee", 2.5)],
             "Kauri Office Supplies": [("A4 paper ream", 12.9), ("Whiteboard markers", 11.5), ("USB-C cable", 19.9), ("Notebook", 6.9)]}

def make_truth(i):
    merchant = rng.choice(list(MERCHANTS))
    items = [(n, round(p, 2)) for n, p in rng.sample(MERCHANTS[merchant], k=rng.randint(2, min(4, len(MERCHANTS[merchant]))))]
    total = round(sum(p for _, p in items), 2)
    return {"receipt_id": f"r{i:02d}", "merchant": merchant, "date": f"2026-{rng.randint(1,9):02d}-{rng.randint(1,28):02d}",
            "items": [{"description": n, "amount": p} for n, p in items], "total": total, "gst": round(total * 3 / 23, 2), "currency": "NZD"}

def render(t, noise=0.0, seed=0):
    """Draw a receipt as a (H, W, 3) uint8 array."""
    fig = plt.figure(figsize=(3.0, 4.0), dpi=100, facecolor="white")
    ax = fig.add_axes([0, 0, 1, 1]); ax.axis("off"); ax.set_xlim(0, 30); ax.set_ylim(0, 40)
    ax.text(15, 37, t["merchant"].upper(), ha="center", family="monospace", weight="bold", fontsize=11)
    ax.text(15, 34.5, f"Date: {t['date']}", ha="center", family="monospace", fontsize=8)
    y = 30
    for it in t["items"]:
        ax.text(2, y, it["description"][:20], family="monospace", fontsize=8)
        ax.text(28, y, f"{it['amount']:.2f}", ha="right", family="monospace", fontsize=8); y -= 3
    ax.plot([2, 28], [y + 1.2, y + 1.2], color="black", lw=0.8)
    ax.text(2, y - 1.5, "TOTAL (NZD)", family="monospace", weight="bold", fontsize=9)
    ax.text(28, y - 1.5, f"{t['total']:.2f}", ha="right", family="monospace", weight="bold", fontsize=9)
    ax.text(2, y - 4.5, "GST 15% incl.", family="monospace", fontsize=8)
    ax.text(28, y - 4.5, f"{t['gst']:.2f}", ha="right", family="monospace", fontsize=8)
    fig.canvas.draw()
    arr = np.asarray(fig.canvas.buffer_rgba())[..., :3].copy(); plt.close(fig)
    if noise:
        arr = np.clip(arr + np.random.default_rng(seed).normal(0, noise, arr.shape), 0, 255).astype(np.uint8)
    return arr

TRUTH = [make_truth(i) for i in range(1, 13)]
IMAGES = {t["receipt_id"]: render(t, noise=(40 if t["receipt_id"] == "r12" else 6), seed=i) for i, t in enumerate(TRUTH)}   # r12 = crumpled, low-quality photo
for t in TRUTH:   # the truth must itself be internally consistent
    assert abs(sum(x["amount"] for x in t["items"]) - t["total"]) < 0.005
print(len(TRUTH), "receipts; image shape:", IMAGES["r01"].shape)

fig, axs = plt.subplots(1, 3, figsize=(8, 3.6))
for ax, rid in zip(axs, ["r01", "r05", "r12"]):
    ax.imshow(IMAGES[rid]); ax.axis("off"); ax.set_title(rid + (" (degraded)" if rid == "r12" else ""))
plt.tight_layout(); plt.show()

## 3 · Build it (a): put the image in a content block
Turn an array into the block shape from section 1. `plt.imsave` encodes PNG without needing PIL. We also compute the token estimate, so image cost is a number you have seen
before the bill arrives.

In [ ]:
from agentkit import get_llm, say, is_live
from agentkit.trace import PRICES
from agentkit.llm import DEFAULT_MODEL

def image_block(arr: np.ndarray) -> dict:
    buf = io.BytesIO(); plt.imsave(buf, arr, format="png")
    return {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": base64.b64encode(buf.getvalue()).decode()}}

def est_image_tokens(arr) -> int:
    h, w = arr.shape[:2]
    return round(w * h / 750)            # docs rule of thumb; confirm in current docs

blk = image_block(IMAGES["r01"])
print("block keys:", list(blk), "| source keys:", list(blk["source"]), "| base64 chars:", len(blk["source"]["data"]))
n_tok = est_image_tokens(IMAGES["r01"])
pin, _ = PRICES["claude-haiku-5-5"]
print(f"~{n_tok} input tokens per receipt image -> ${n_tok * pin / 1e6:.6f} each at the cheap tier; x3000/month = ${3000 * n_tok * pin / 1e6:.2f} (image input only)")
assert blk["type"] == "image" and base64.b64decode(blk["source"]["data"])[:4] == b"\x89PNG"

## 4 · Build it (b): the schema is the contract
We ask for JSON and then **validate it ourselves**. The pydantic model encodes what we know about receipts, and the `model_validator` encodes *redundancy*: the document says the
same thing in several places (items, total, GST), so a misread digit usually breaks an equation. This is the cheapest error detector you will ever write.

In [ ]:
from datetime import date
from pydantic import BaseModel, Field, field_validator, model_validator, ValidationError

class LineItem(BaseModel):
    description: str = Field(min_length=1)
    amount: float = Field(ge=0)

class Receipt(BaseModel):
    merchant: str = Field(min_length=2)
    date: str
    items: list[LineItem] = Field(min_length=1)
    total: float = Field(gt=0)
    gst: float = Field(ge=0)
    currency: str = "NZD"

    @field_validator("date")
    @classmethod
    def _date_ok(cls, v):
        d = date.fromisoformat(v)                     # raises ValueError on garbage
        if not (date(2020, 1, 1) <= d <= date(2026, 12, 31)):
            raise ValueError("date outside plausible range")
        return v

    @model_validator(mode="after")
    def _consistent(self):
        if abs(sum(i.amount for i in self.items) - self.total) > 0.011:
            raise ValueError(f"items sum {sum(i.amount for i in self.items):.2f} != total {self.total:.2f}")
        if abs(self.gst - round(self.total * 3 / 23, 2)) > 0.02:
            raise ValueError(f"gst {self.gst:.2f} is not 15% GST-inclusive of total {self.total:.2f}")
        return self

class Extraction(BaseModel):
    receipt: Receipt
    confidence: dict[str, float]          # model's own per-field self-report in [0, 1]

good = {k: v for k, v in TRUTH[0].items() if k != "receipt_id"}
Receipt(**good)                                         # passes
for name, bad in {"misread total": {**good, "total": good["total"] + 5.0}, "impossible date": {**good, "date": "2026-13-45"}}.items():
    try: Receipt(**bad); raise AssertionError("should have failed")
    except ValidationError as e: print(f"{name:16s} -> rejected: {e.errors()[0]['msg'][:70]}")

## 5 · Use it: an extraction pipeline with retry and routing
Now the model. **Offline, the replies are scripted** by a function playing the vision model: it never sees pixels, it returns what a real model plausibly returned for each image,
including four realistic failures (below). Live, the same code sends the real image. The pipeline logic is the real, tested part.

| receipt | simulated model behaviour |
|---|---|
| r03 | misreads the total by 5.00, so the line items no longer sum to it |
| r07 | reads the date correctly but **reports low confidence**, since the photo is smudged there |
| r09 | first reply is malformed JSON (trailing comma and chatter); correct on retry |
| r11 | **confidently wrong** merchant name that no equation can catch |
| r12 | degraded photo: all-low confidence |

In [ ]:
EXTRACT_PROMPT = ("Extract this receipt as JSON with keys: receipt {merchant, date (YYYY-MM-DD), items[{description, amount}], total, gst, currency}, "
                  "and confidence {merchant, date, total} each in [0,1]. Output JSON only.")
TRUTH_BY_ID = {t["receipt_id"]: t for t in TRUTH}

def simulated_model_reply(rid: str, attempt: int) -> str:
    t = {k: v for k, v in TRUTH_BY_ID[rid].items() if k != "receipt_id"}
    t = json.loads(json.dumps(t))
    conf = {"merchant": 0.98, "date": 0.96, "total": 0.97}
    if rid == "r03":
        t["total"] = round(t["total"] - 5.0, 2); t["gst"] = round(t["total"] * 3 / 23, 2); conf["total"] = 0.93   # plausible, self-consistent total... but items disagree
    if rid == "r07": conf["date"] = 0.55
    if rid == "r11": t["merchant"] = "Kauri Office Supply Co"; conf["merchant"] = 0.95
    if rid == "r12": conf = {"merchant": 0.5, "date": 0.4, "total": 0.45}
    body = json.dumps({"receipt": t, "confidence": conf})
    if rid == "r09" and attempt == 0:
        return "Sure! Here is the data you asked for:\n" + body[:-1] + ",}"       # invalid JSON
    return body

from agentkit import PolicyLLM, Reply
def vision_policy(messages, system, tools):
    first = messages[0]["content"]
    rid = re.search(r"receipt_id: (r\d+)", next(b["text"] for b in first if b["type"] == "text")).group(1)
    attempt = sum(1 for m in messages if m["role"] == "assistant")
    txt = simulated_model_reply(rid, attempt)
    return Reply([{"type": "text", "text": txt}], "end_turn", {"input_tokens": 360 + 25 * attempt, "output_tokens": len(txt) // 4})

llm = get_llm(policy=vision_policy)

def parse_json(text: str):
    m = re.search(r"\{.*\}", text, re.S)
    if not m: raise ValueError("no JSON object found")
    return json.loads(m.group(0))                      # strict: trailing commas still fail

def extract(rid: str, max_attempts: int = 2):
    """-> (Extraction | None, attempts_used, last_error)"""
    messages = [{"role": "user", "content": [image_block(IMAGES[rid]), {"type": "text", "text": f"receipt_id: {rid}\n{EXTRACT_PROMPT}"}]}]
    err = None
    for attempt in range(max_attempts):
        reply = llm.chat(messages, system="You extract data from receipts. Never guess unreadable fields.")
        messages.append({"role": "assistant", "content": reply.content})
        try:
            return Extraction.model_validate(parse_json(reply.text)), attempt + 1, None
        except (ValueError, ValidationError) as e:            # JSONDecodeError and pydantic errors are ValueErrors
            err = str(e)[:160]
            messages.append({"role": "user", "content": f"Your output was invalid: {err}\nReturn corrected JSON only."})
    return None, max_attempts, err

res = {t["receipt_id"]: extract(t["receipt_id"]) for t in TRUTH}
for rid in ["r01", "r03", "r09"]:
    ex, n, err = res[rid]
    print(rid, "| attempts:", n, "| parsed:", ex is not None, "|", (err or "ok")[:90])

Note what happened to **r03**: the model's reply is *valid JSON that fails our schema* (items sum ≠ total), so the retry loop fed the error back. Whether the retry fixes it depends on
the model; the scripted model repeats its mistake, which is realistic, because a model that misread a digit once usually misreads it again from the same pixels. **Retries fix
format errors; they rarely fix perception errors.** That is what the router is for.

## 6 · Confidence routing
Three signals decide `auto` vs `review`:
1. **Hard checks** passed (parsed and consistent). Failure goes straight to a human.
2. **Self-reported confidence** (min over fields) at or above a threshold τ.
3. (Optional, expensive) a second pass or a second model disagreeing, which we leave as an exercise.

Self-reported confidence is only weakly calibrated: treat it as a *ranking* signal, and tune τ on labelled data, never by feel.

In [ ]:
FIELDS = ["merchant", "date", "total"]

def route(ex, tau=0.85):
    if ex is None: return "review", "unparseable/inconsistent"
    low = {k: v for k, v in ex.confidence.items() if v < tau}
    return ("review", f"low confidence {low}") if low else ("auto", "ok")

def field_correct(ex, rid):
    t = TRUTH_BY_ID[rid]
    return {"merchant": ex.receipt.merchant == t["merchant"], "date": ex.receipt.date == t["date"], "total": abs(ex.receipt.total - t["total"]) < 0.005}

def evaluate(tau):
    auto_ok = auto_bad = reviewed = 0
    bad_ids = []
    for rid, (ex, n, err) in res.items():
        dest, why = route(ex, tau)
        if dest == "review": reviewed += 1; continue
        if all(field_correct(ex, rid).values()): auto_ok += 1
        else: auto_bad += 1; bad_ids.append(rid)
    return {"tau": tau, "auto_rate": (auto_ok + auto_bad) / len(res), "auto_error_rate": auto_bad / max(1, auto_ok + auto_bad), "review": reviewed, "bad_auto": bad_ids}

for tau in (0.5, 0.85, 0.96):
    print(evaluate(tau))

## 7 · Look at the trade-off
Sweep τ and plot **what fraction is automated** against **how many automated records are wrong**. A threshold is a business decision: a wrong ledger entry costs a clerk
(say) 10 minutes to find later; a review costs about 1 minute of triage. Put your own costs on the axes.

In [ ]:
taus = np.round(np.arange(0.3, 1.0, 0.05), 2)
rows = [evaluate(t) for t in taus]
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(taus, [r["auto_rate"] for r in rows], "o-", label="automation rate")
ax.plot(taus, [r["auto_error_rate"] for r in rows], "s--", label="error rate among auto-approved")
ax.set_xlabel("confidence threshold τ"); ax.set_ylim(-0.02, 1.05); ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

REVIEW_MIN, FIX_LATER_MIN = 1.0, 10.0
def clerk_minutes(tau):
    r = evaluate(tau); n = len(res)
    return r["review"] * REVIEW_MIN + len(r["bad_auto"]) * FIX_LATER_MIN
best_tau = min((0.5, 0.85, 0.96), key=clerk_minutes)
print({t: clerk_minutes(t) for t in (0.5, 0.85, 0.96)}, "-> chosen tau:", best_tau)
print("(12 receipts is far too few to tune tau for real: use a few hundred labelled ones.)")

if not is_live():     # strict numbers hold for the scripted model only
    r = evaluate(0.85)
    assert res["r03"][0] is None, "inconsistent extraction must not parse"
    assert res["r09"][0] is not None and res["r09"][1] == 2, "format error should be fixed by one retry"
    assert r["bad_auto"] == ["r11"], "only the confidently-wrong merchant slips through at tau=0.85"
    assert evaluate(0.85)["review"] == 3                  # r03, r07, r12

## 8 · Break it
**8a. The confident error.** r11 sailed through at τ ≤ 0.9 (it was caught at 0.96 only because we scripted its confidence at 0.95; a model that says 0.99 passes any τ): a wrong-but-plausible merchant has no redundancy to contradict it and the model was sure. No threshold fixes this.
What does? (i) A **lookup against a known list** (vendor master data: "Kauri Office Supply Co" is not an approved vendor, so review). (ii) A **second independent read** (different model
or prompt) and routing on disagreement. (iii) **Random audit sampling** of auto-approved records, so you *measure* your silent-error rate instead of assuming it.

**8b. Garbage in, plausible out.** Models rarely say "I can't read this" unless asked. The system prompt above says *"Never guess unreadable fields"* and the schema has no `null`, so the
model is forced to either guess or fail validation. Better schema design: make uncertain fields `Optional` plus an explicit `unreadable: list[str]`.

In [ ]:
APPROVED_VENDORS = set(MERCHANTS)

def route_v2(ex, tau=0.85):
    dest, why = route(ex, tau)
    if dest == "auto" and ex.receipt.merchant not in APPROVED_VENDORS:
        return "review", "vendor not in approved list"
    return dest, why

dest, why = route_v2(res["r11"][0]); print("r11 ->", dest, "|", why)
assert dest == "review" or is_live()

audit_rng = random.Random(1)
auto_ids = [rid for rid, (ex, _, _) in res.items() if route(ex)[0] == "auto"]
audit = audit_rng.sample(auto_ids, k=4)
print("random audit sample of auto-approved:", audit, "(clerks check these against the image even though the system was 'sure')")

## 9 · OCR vs native vision: which tool for which document?

| | Classic OCR (+ rules/layout model) | Native vision LLM |
|---|---|---|
| Clean, fixed-layout forms at huge volume | **Strong**: cheap, fast, deterministic, no hallucination of absent text | Overkill; slower and costlier per page |
| Messy photos, varied layouts, handwriting | Brittle: needs per-template rules | **Strong**: layout and context understood jointly |
| Tables, charts, figures | Needs a separate layout/table model | Often decent; verify numbers |
| Failure mode | Garbled characters (`O`↔`0`, `5`↔`S`): *visible* noise | Fluent, plausible, *wrong* values: invisible without checks |
| Auditability | Returns character boxes and per-character confidence | Free-text reasoning only; no pixel-level provenance |
| Typical cost shape | Per page, cheap | Per token: images + output tokens |

A common production pattern is a **hybrid**: OCR for text and boxes, vision model for fields the OCR could not parse, schema validation over both. Let us see OCR's failure mode
numerically: simulate character confusions at rate *p* and a regex parser, then measure exact-total accuracy.

In [ ]:
CONFUSE = {"0": "O", "1": "l", "5": "S", "8": "B", "2": "Z"}
def fake_ocr(text: str, p: float, r: random.Random) -> str:
    return "".join(CONFUSE[c] if c in CONFUSE and r.random() < p else c for c in text)

def ocr_parse_total(text: str):
    m = re.search(r"TOTAL[^\d]*([\d]+\.\d{2})", text)
    return float(m.group(1)) if m else None

def ocr_accuracy(p, trials=300):
    r = random.Random(3); ok = parsed = 0
    for k in range(trials):
        t = TRUTH[k % len(TRUTH)]
        page = fake_ocr(f"{t['merchant']}\nTOTAL (NZD) {t['total']:.2f}\nGST {t['gst']:.2f}", p, r)
        v = ocr_parse_total(page)
        parsed += v is not None; ok += (v is not None and abs(v - t["total"]) < 0.005)
    return parsed / trials, ok / trials

for p in (0.0, 0.05, 0.15):
    parsed, ok = ocr_accuracy(p); print(f"char-confusion rate {p:.2f}: regex found a total in {parsed:.0%}, exactly right in {ok:.0%}")
assert ocr_accuracy(0.0)[1] == 1.0 and ocr_accuracy(0.15)[1] < ocr_accuracy(0.05)[1] < 1.0

Notice the OCR failure is **loud** (the parse fails or the string is visibly corrupt), which makes it easy to route; the vision-model failure is **quiet**, which is why section 4's
consistency checks matter more, not less, when you move to a vision model.

## 10 · Audio in one page: ASR and TTS
Speech is handled by *separate* models that sit around the LLM, and the contract is text:

```
mic → [ASR: speech-to-text] → text → [LLM + tools] → text → [TTS: text-to-speech] → speaker
        WER, latency (streaming)                                 voice quality, time-to-first-audio
```

* **ASR** is scored by **word error rate**: `WER = (S + D + I) / N` (substitutions, deletions, insertions over reference words), the word-level edit distance. Open-weight systems
  such as Whisper (Radford et al., 2022, arXiv:2212.04356) are a common baseline. Check which modalities your model API accepts natively; do not assume it takes raw audio.
* **Latency compounds**: ASR + LLM time-to-first-token + TTS time-to-first-audio. Streaming every stage is what makes a voice agent feel conversational (A09 covers TTFT).
* **Error propagation**: an ASR mistake on a name or number is invisible to the LLM. For high-stakes fields, read them back to the caller ("that's *four-one-seven*, correct?").

In [ ]:
def wer(ref: str, hyp: str) -> float:
    r, h = ref.lower().split(), hyp.lower().split()
    d = np.zeros((len(r) + 1, len(h) + 1), dtype=int)
    d[:, 0] = np.arange(len(r) + 1); d[0, :] = np.arange(len(h) + 1)
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            d[i, j] = min(d[i-1, j] + 1, d[i, j-1] + 1, d[i-1, j-1] + (r[i-1] != h[j-1]))
    return d[len(r), len(h)] / len(r)

ref = "please book a taxi to the airport for four fifteen pm"
print("clean ASR  :", wer(ref, ref))
print("noisy ASR  :", round(wer(ref, "please book a taxi to the air port for fourteen fifteen pm"), 3))
assert wer(ref, ref) == 0 and abs(wer("a b c d", "a x c") - 0.5) < 1e-9     # 1 substitution + 1 deletion over 4 words

## 11 · Cost and latency of the whole pipeline
Add it up with the tracer-style bookkeeping you already know: tokens in/out per receipt (including retries), at a cheap and a strong tier.

In [ ]:
tot_in = tot_out = 0
for t in TRUTH:
    ex, n, err = res[t["receipt_id"]]
    tot_in += n * (est_image_tokens(IMAGES[t["receipt_id"]]) + 120); tot_out += n * 160      # prompt text ~120 tok, JSON ~160 tok
for m in ("claude-haiku-5-5", "claude-sonnet-5-5"):
    pin, pout = PRICES[m]
    per = (tot_in * pin + tot_out * pout) / 1e6 / len(TRUTH)
    print(f"{m:20s} ${per:.5f}/receipt -> ${per * 3000:.2f} per 3,000 receipts")
print("A clerk typing a receipt: ~2 min. Even the strong tier is orders of magnitude cheaper; the real cost is the errors you do not catch.")

## 12 · Exercises
**Practice (30 min)**
1. Add a `payment_method: Optional[str]` field to `Receipt` and make `route` send receipts with `payment_method is None` to review. Re-run the sweep.
2. Add a validator that rejects dates in the future relative to a `today` argument. Which receipt fails if today is `2026-03-01`?

**Stretch (60 min)**
3. Implement a **second-opinion check**: call a second scripted reply for each receipt (make r11's second read say "Kauri Office Supplies") and route on field disagreement. Show that r11 is now caught without a vendor list.
4. Estimate the **break-even**: if clerks need 2 min per receipt, 1 min per review and 10 min per undetected error, at what auto-error rate does automating stop paying off?

**Challenge (2 h, optional LIVE)**
5. Build 30 receipts with harder rendering (rotation, lower resolution, a crumpled-paper noise texture), run LIVE, and report field accuracy, automation rate and calibration (bucket confidence into deciles and plot accuracy per bucket).

In [ ]:
# Starter for exercise 2 — fill in and make the assertions pass
def not_in_future(r: Receipt, today: str) -> bool:
    # TODO: return True if r.date <= today
    raise NotImplementedError

# future = [rid for rid, (ex, _, _) in res.items() if ex and not not_in_future(ex.receipt, "2026-03-01")]
# assert future == [t["receipt_id"] for t in TRUTH if t["date"] > "2026-03-01" and res[t["receipt_id"]][0]]
# print("receipts dated after 2026-03-01:", future)

## ✅ Checkpoint
1. Why does a pydantic `model_validator` that checks "items sum to total" help *more* with a vision model than with OCR?
2. Retrying with the validation error in the prompt fixed r09 but not r03. Why the difference?
3. A teammate sets τ = 0.99 so "nothing wrong gets through". What happens, and what still slips through?
4. Why should a voice assistant read back account numbers even if ASR WER is 3%?

<details><summary>Answers</summary>

1. Vision-model errors are fluent and plausible, so you cannot spot them by eye; redundancy inside the document is an independent check that costs nothing.
2. r09 was a *format* error (the model knew the answer, just wrote it badly); r03 was a *perception* error, and the same pixels yield the same misread. Retries fix the former, routing handles the latter.
3. Automation collapses (almost everything goes to review), defeating the purpose, and confidence is not correctness: a model that is confidently wrong (like r11, if it had said 0.99) still passes any threshold below 1. Use audits, vendor lists, or a second read.
4. A 3% average hides that errors concentrate on names and digits, which are exactly the high-stakes tokens, and the LLM cannot know the ASR got them wrong.
</details>

## 📌 Key takeaways
- Images and PDFs are **content blocks** in the user turn; image cost is input tokens (about pixels/750, an estimate to confirm in the docs).
- Extract into a **schema**, validate with **cross-field consistency**, retry format errors, and **route** perception errors and low confidence to humans.
- Self-reported confidence is a ranking signal, not a probability. Tune τ on labelled data, audit the auto-approved, and add external checks (vendor lists, second reads).
- OCR fails loudly, vision fails quietly. Choose by document variability and volume; hybrids are common.
- Audio is ASR + LLM + TTS around a text contract; measure WER and compounded latency.

## 📚 Further reading
- Anthropic docs: *Vision* and *PDF support* (content-block formats, size limits, token estimates)
- Radford et al., *Robust Speech Recognition via Large-Scale Weak Supervision* (Whisper, 2022), arXiv:2212.04356
- pydantic docs: *Validators* (`field_validator`, `model_validator`)

**Next → A09 · Serving, Inference Optimization & Cost** (what that image and that retry loop cost, and how to make them cheaper)
